In [ ]:
subset the benchamrk to enhncers overlapping my peaks first

In [ ]:
# first overlap your peaks with perturbation enhancers
# to get a universe of testable peak-gene pairs

# build your peak-gene pairs
my_pairs <- finalres %>%
    select(chr, start, end, top_gene) %>%
    distinct()

# build perturbation pairs (adjust col names to match your data)
perturb_pairs <- perturb %>%
    select(enh_chr, enh_start, enh_end, gene, validated) %>%
    distinct()

# overlap peaks with perturbation enhancers using GRanges
my_gr <- GRanges(seqnames=my_pairs$chr, 
                 ranges=IRanges(my_pairs$start, my_pairs$end))
perturb_gr <- GRanges(seqnames=perturb_pairs$enh_chr,
                      ranges=IRanges(perturb_pairs$enh_start, perturb_pairs$enh_end))

hits <- findOverlaps(my_gr, perturb_gr)

# for each overlap, check if gene matches
overlap_df <- data.frame(
    my_gene = my_pairs$top_gene[queryHits(hits)],
    perturb_gene = perturb_pairs$gene[subjectHits(hits)],
    validated = perturb_pairs$validated[subjectHits(hits)]
) %>%
    mutate(gene_match = my_gene == perturb_gene)

# contingency table
tab <- table(
    in_my_links = overlap_df$gene_match,
    experimentally_validated = overlap_df$validated
)
print(tab)
fisher.test(tab)